In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata
import scipy

import os

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
import session_info
session_info.show()

/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)


In [3]:
sc.settings.set_figure_params(dpi=120)

In [4]:
import importlib.util
import sys
spec = importlib.util.spec_from_file_location("module.name", "/rfs/project/rfs-iCNyzSAaucw/kk837/function/python/utils.py")
utils = importlib.util.module_from_spec(spec)
sys.modules["module.name"] = utils
spec.loader.exec_module(utils)

/rfs/project/rfs-iCNyzSAaucw/kk837/function/python/utils.py:356: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if sc.__version__.startswith("1.4"):


# Variables

In [5]:
data_object_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025'
latent_space_condition = 'n-layers-3'
latent_space = f'scVI_latent_{latent_space_condition}'

# Read in adata

In [6]:
%%time
adata = sc.read_h5ad(f'{data_object_dir}/epicardium-mixture_b2c_cells_filtered_raw.h5ad')

CPU times: user 185 ms, sys: 1.75 s, total: 1.93 s
Wall time: 31.8 s


In [7]:
# delete previous latent space and clustering results
obs_col = adata.obs.columns.copy()
obs_col = [col for col in obs_col if '_leiden_' in col]
adata.obs.drop(columns=obs_col,inplace=True)

# delete obsm
obsm_keys = list(adata.obsm.keys())
for key in obsm_keys:
    if 'scVI' in key:
        del adata.obsm[key]
        
# delete uns
uns_keys = list(adata.uns.keys())
for key in uns_keys:
    if 'scVI' in key:
        del adata.uns[key]

adata

AnnData object with n_obs × n_vars = 45632 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_colors', 'donor_colors', 'library_colors', 'neighbors', 'spatial', 'umap'
    obsm: 'X_umap_wo-batch-correction', 'spatial', 'spatial_cropped_150_buffer'
    obsp: 'connectivities', 'distances'

# Add scVI latent space

In [8]:
adata_latent = sc.read_h5ad(f'{data_object_dir}/scVI/latent_variables/epicardium-mixture_correcting-donor_n-layers-3.h5ad')
adata_latent

AnnData object with n_obs × n_vars = 45632 × 50

In [9]:
adata.obsm[latent_space] = adata_latent[adata.obs_names].X.copy()

# Clustering

In [10]:
%%time
sc.pp.neighbors(adata, use_rep=latent_space, n_neighbors=15)

/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 24.1 s, sys: 208 ms, total: 24.3 s
Wall time: 30.1 s


In [11]:
%%time
leiden_keys=[]
for res in tqdm([0.2,0.3,0.4,0.5,0.8,1.0,1.5,2.0,3.0,4.0]):
    leiden_key = f"leiden_{res}"
    key_added=f"{latent_space}_{leiden_key}"
    sc.tl.leiden(adata,resolution=res,key_added=key_added,n_iterations=2)
    leiden_keys.append(key_added)

  0%|          | 0/10 [00:00<?, ?it/s]<timed exec>:5: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
100%|██████████| 10/10 [01:05<00:00,  6.54s/it]

CPU times: user 1min 4s, sys: 544 ms, total: 1min 5s
Wall time: 1min 5s


# Save

In [12]:
adata.write(f'{data_object_dir}/epicardium-mixture_b2c_cells_filtered_raw.h5ad')